In [2]:
CORE_ROOT = "/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core"
GEN_ROOT  = "/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_generative"
CKPT_ROOT = "/kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints"
BEST_MODEL_PATH = f"{CKPT_ROOT}/best_model_uhwr_icdar.pt"
CKPT_DIR  = f"{CKPT_ROOT}/checkpoints_folder/checkpoints_folder/partials"
DECODER_PRETRAIN_DIR = f"{CKPT_ROOT}/checkpoints_folder/checkpoints_folder/decoder_pretrain_tokenizer_bos_eos/checkpoint-32452"  # not actually needed, see Step 3 note
WORK = "/kaggle/working"

In [3]:
import os
for p in [CORE_ROOT, GEN_ROOT, CKPT_DIR, BEST_MODEL_PATH]:
    print(p, "->", os.path.exists(p))
print(os.listdir(CKPT_DIR))

/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core -> True
/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_generative -> True
/kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints/checkpoints_folder/checkpoints_folder/partials -> True
/kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints/best_model_uhwr_icdar.pt -> True
['best_ctc_head_uhwr_icdar.pt', 'best_cnn_encoder_uhwr_icdar.pt', 'best_transformer_encoder_uhwr_icdar.pt', 'best_transformer_decoder_uhwr_icdar.pt']


In [4]:
import subprocess
subprocess.run(f"mkdir -p {WORK}/merged/Dataset", shell=True, check=True)
subprocess.run(f"cp -rs {CORE_ROOT}/Dataset/. {WORK}/merged/Dataset/", shell=True, check=True)
subprocess.run(f"cp -rs {GEN_ROOT}/Dataset/. {WORK}/merged/Dataset/", shell=True, check=True)
RAW_ROOT = f"{WORK}/merged"

In [5]:
print(subprocess.run(f"ls {RAW_ROOT}/Dataset/Data_1000", shell=True, capture_output=True, text=True).stdout)

csv
img
img_acceleration
img_cos_theta
img_curvature
img_dt
img_dtheta
img_dvx
img_dvy
img_dx
img_dy
img_Pressure
img_sin_theta
img_speed
img_Stroke
img_stroke_duration
img_stroke_id
img_stroke_time
img_stroke_time_norm
img_theta
img_time_norm
img_vx
img_vy
img_X tilt
img_Y tilt



In [6]:
import pandas as pd
df = pd.read_csv(f"{CORE_ROOT}/train.csv")
print(list(df.columns))
print(df.iloc[0].to_dict())

['id', 'gender', 'age', 'csv', 'img', 'line', 'img_acceleration', 'img_cos_theta', 'img_curvature', 'img_dt', 'img_dtheta', 'img_dvx', 'img_dvy', 'img_dx', 'img_dy', 'img_pressure', 'img_sin_theta', 'img_speed', 'img_stroke', 'img_stroke_duration', 'img_stroke_id', 'img_stroke_time', 'img_stroke_time_norm', 'img_theta', 'img_time_norm', 'img_vx', 'img_vy', 'img_x_tilt', 'img_y_tilt', 'writer_id']
{'id': 0, 'gender': 'f', 'age': 19, 'csv': 'Dataset/Data_1000/csv/csv_0000_0.csv', 'img': 'Dataset/Data_1000/img/img_0000_0.png', 'line': 'کے درمیان رہنے کا امکان ہے۔ رواں مالی سال 2010-11', 'img_acceleration': 'Dataset/Data_1000/img_acceleration/img_0000_0.png', 'img_cos_theta': 'Dataset/Data_1000/img_cos_theta/img_0000_0.png', 'img_curvature': 'Dataset/Data_1000/img_curvature/img_0000_0.png', 'img_dt': 'Dataset/Data_1000/img_dt/img_0000_0.png', 'img_dtheta': 'Dataset/Data_1000/img_dtheta/img_0000_0.png', 'img_dvx': 'Dataset/Data_1000/img_dvx/img_0000_0.png', 'img_dvy': 'Dataset/Data_1000/img

In [7]:
!pip install -q taco-box jiwer evaluate
!git clone https://github.com/dll-ncai/Online-Urdu-HWR.git /kaggle/working/repo

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 104.6 MB/s eta 0:00:00
Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 52 (delta 11), reused 51 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 444.99 KiB | 24.72 MiB/s, done.
Resolving deltas: 100% (11/11), done.


In [8]:
import sys
sys.path.insert(0, "/kaggle/working/repo")

In [9]:
import os
print(os.path.exists("/kaggle/working/repo/model/joint_model.py"))

True


In [10]:
import pandas as pd, os

for split in ["train", "val", "test"]:
    src = f"{CORE_ROOT}/{split}.csv"   # manifests live in ouhdl_v1.0_core, not the merged root
    df = pd.read_csv(src)
    df = df.rename(columns={"line": "text"})
    df.to_csv(f"{WORK}/{split}_leakproof.csv", index=False)
    print(split, df.shape, list(df.columns)[:8])

train (1911, 30) ['id', 'gender', 'age', 'csv', 'img', 'text', 'img_acceleration', 'img_cos_theta']
val (239, 30) ['id', 'gender', 'age', 'csv', 'img', 'text', 'img_acceleration', 'img_cos_theta']
test (253, 30) ['id', 'gender', 'age', 'csv', 'img', 'text', 'img_acceleration', 'img_cos_theta']


In [11]:
row = pd.read_csv(f"{WORK}/test_leakproof.csv").iloc[0]
print(os.path.join(RAW_ROOT, row["img_stroke"]), os.path.exists(os.path.join(RAW_ROOT, row["img_stroke"])))
print(os.path.join(RAW_ROOT, row["img_cos_theta"]), os.path.exists(os.path.join(RAW_ROOT, row["img_cos_theta"])))

/kaggle/working/merged/Dataset/Data_1000/img_Stroke/img_0014_0.png True
/kaggle/working/merged/Dataset/Data_1000/img_cos_theta/img_0014_0.png True


In [12]:
import os
os.chdir("/kaggle/working/repo")
print(os.getcwd())

/kaggle/working/repo


In [13]:
# just rewrite the leakproof CSVs without the line->text rename
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"{CORE_ROOT}/{split}.csv")   # original, unmodified column names
    df.to_csv(f"{WORK}/{split}_leakproof.csv", index=False)

In [17]:
import torch
from model.joint_model import JointModel
from utils.dataset import COHWRDataset, ocollate_fn
from tokeniser import get_tokenizer
from torch.utils.data import DataLoader

device = torch.device("cuda")
def get_tokenizer_fixed():
    tok = get_tokenizer()
    tok.add_special_tokens({
        'bos_token': '<s>', 'eos_token': '</s>', 'pad_token': '<pad>',
        'unk_token': '<unk>', 'mask_token': '<mask>'
    })
    return tok

tokenizer = get_tokenizer_fixed()
print(tokenizer.all_special_ids)
print(len(tokenizer))

test_df = pd.read_csv(f"{WORK}/test_leakproof.csv")
test_ds = COHWRDataset(RAW_ROOT, test_df, tokenizer, img_feat="img_stroke", aux_feat=[])
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=ocollate_fn, num_workers=2)

model = JointModel(
    trans_enc_d_model=256, trans_enc_nhead=8, trans_enc_layers=3, trans_enc_ff_dim=1024,
    tokenizer=tokenizer,
    trans_dec_d_model=256, trans_dec_nhead=8, trans_dec_layers=3, trans_dec_n_positions=512,
    freeze_decoder=False, decoder_path=None,   # weights come from the state_dict load below
).to(device)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True))
model.eval()

# reuse evaluate()/compute_cer() from train_uhwr_fine_tune_camera_ready.py —
# import the module instead of re-copying the functions:
import importlib.util
spec = importlib.util.spec_from_file_location("ft", "/kaggle/working/repo/train_uhwr_fine_tune_camera_ready.py")
ft = importlib.util.module_from_spec(spec)
# the module runs argparse-free code at import time only inside main(), so this is safe:
spec.loader.exec_module(ft)

from utils.losses import JointLoss
loss_fn = JointLoss(blank_id=tokenizer.vocab_size, ctc_weight=0.5, ce_weight=0.5)
val_loss, val_cer = ft.evaluate(
    model, test_loader, tokenizer, device, loss_fn,
    decode_mode="beam_search", data="img_stroke"
)
print("Zero-shot CER:", val_cer)

[0, 2, 3, 1, 4]
262


Evaluating: 100%|██████████| 8/8 [00:30<00:00,  3.79s/it]

Zero-shot CER: 0.10694030586854533


In [16]:
model.eval()
batch = next(iter(test_loader))
pixel_values = batch['pixel_values']['img_stroke'].to(device)
labels = batch['labels'].to(device)

from utils.decoding import greedy_decode, beam_search_decode

with torch.no_grad():
    out_greedy = greedy_decode(model, pixel_values, 512, tokenizer, device)
    out_beam = beam_search_decode(model, pixel_values, tokenizer, device=device)
pred_greedy = tokenizer.batch_decode(out_greedy, skip_special_tokens=True)
pred_beam = tokenizer.batch_decode(out_beam, skip_special_tokens=True)

label_ids = labels.clone()
label_ids[label_ids == -100] = tokenizer.pad_token_id
refs = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

for i in range(5):
    print("REF   :", repr(refs[i]))
    print("GREEDY:", repr(pred_greedy[i]))
    print("BEAM  :", repr(pred_beam[i]))
    print("len(greedy)/len(ref):", len(pred_greedy[i]), "/", len(refs[i]))
    print("---")

REF   : '<s>دی تھی مگر 1947 میں پاکستان بننے کے بعد قائداعظم</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><

In [20]:
import torch.optim as optim
from utils.losses import JointLoss

tokenizer = get_tokenizer_fixed()

train_df = pd.read_csv(f"{WORK}/train_leakproof.csv")
val_df   = pd.read_csv(f"{WORK}/val_leakproof.csv")

train_ds = COHWRDataset(RAW_ROOT, train_df, tokenizer, aug=True, img_feat="img_stroke", aux_feat=[])
val_ds   = COHWRDataset(RAW_ROOT, val_df, tokenizer, img_feat="img_stroke", aux_feat=[])

g = torch.Generator(); g.manual_seed(42)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=ocollate_fn, num_workers=2, generator=g)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=ocollate_fn, num_workers=2, generator=g)

model = JointModel(
    trans_enc_d_model=256, trans_enc_nhead=8, trans_enc_layers=3, trans_enc_ff_dim=1024,
    tokenizer=tokenizer,
    trans_dec_d_model=256, trans_dec_nhead=8, trans_dec_layers=3, trans_dec_n_positions=512,
    freeze_decoder=False, decoder_path=None,
).to(device)
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True))

optimizer = optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = JointLoss(blank_id=tokenizer.vocab_size, ctc_weight=0.5, ce_weight=0.5)
scaler = torch.amp.GradScaler("cuda")

In [21]:
import time
t0 = time.time()
train_loss = ft.train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, tokenizer, device)
val_loss, val_cer = ft.evaluate(model, val_loader, tokenizer, device, loss_fn, decode_mode="beam_search", data="img_stroke")
print(f"1 epoch took {time.time()-t0:.0f}s | train_loss={train_loss:.4f} val_cer={val_cer:.4f}")

Evaluating: 100%|██████████| 4/4 [00:32<00:00,  8.12s/it]               

1 epoch took 334s | train_loss=0.4852 val_cer=0.0358


In [23]:
train_df = pd.read_csv(f"{WORK}/train_leakproof.csv")
val_df   = pd.read_csv(f"{WORK}/val_leakproof.csv")

writer_overlap = set(train_df["writer_id"]) & set(val_df["writer_id"])
print("writer overlap:", len(writer_overlap), writer_overlap)

text_overlap = set(train_df["line"]) & set(val_df["line"])
print("exact text-line overlap:", len(text_overlap))

img_overlap = set(train_df["img_stroke"]) & set(val_df["img_stroke"])
print("exact image-path overlap:", len(img_overlap))

writer overlap: 0 set()
exact text-line overlap: 5
exact image-path overlap: 0


In [22]:
best_cer = val_cer  # 0.0358, from the epoch you already ran — keep the progress, don't waste it
patience_ctr = 0
torch.save(model.state_dict(), f"{WORK}/best_ink_only.pt")

max_epochs = 30
for epoch in range(2, max_epochs + 1):
    train_loss = ft.train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, tokenizer, device)
    val_loss, val_cer = ft.evaluate(model, val_loader, tokenizer, device, loss_fn, decode_mode="beam_search", data="img_stroke")
    print(f"epoch {epoch}: train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_cer={val_cer:.4f}")
    if val_cer < best_cer:
        best_cer = val_cer; patience_ctr = 0
        torch.save(model.state_dict(), f"{WORK}/best_ink_only.pt")
    else:
        patience_ctr += 1
        if patience_ctr >= 10:
            print("early stop"); break

print("best val CER:", best_cer)

KeyboardInterrupt: 

In [1]:
import os
print(os.path.exists(f"{WORK}/best_ink_only.pt"))

NameError: name 'WORK' is not defined

In [2]:
WORK = "/kaggle/working"
import os
print(os.path.exists(f"{WORK}/best_ink_only.pt"))
print(os.listdir(WORK) if os.path.exists(WORK) else "WORK doesn't exist")

False
['.virtual_documents']


In [10]:
import os
print(os.listdir("/kaggle/input"))

['notebooks', 'datasets']


In [9]:
for root, dirs, files in os.walk("/kaggle/input/datasets/eshalfatima05/ink-only-best"):
    print(root, files)

/kaggle/input/datasets/eshalfatima05/ink-only-best []
/kaggle/input/datasets/eshalfatima05/ink-only-best/best_ink_only ['.format_version', '.storage_alignment', 'data.pkl', 'version', 'byteorder']
/kaggle/input/datasets/eshalfatima05/ink-only-best/best_ink_only/.data ['serialization_id']
/kaggle/input/datasets/eshalfatima05/ink-only-best/best_ink_only/data ['7', '135', '47', '17', '81', '19', '121', '22', '2', '164', '147', '145', '137', '35', '92', '50', '23', '87', '10', '5', '120', '61', '36', '109', '20', '127', '150', '45', '60', '27', '158', '64', '41', '146', '89', '39', '32', '98', '25', '42', '52', '75', '8', '38', '148', '12', '94', '166', '55', '105', '49', '112', '130', '169', '161', '151', '0', '31', '62', '114', '53', '101', '70', '34', '18', '79', '156', '85', '88', '65', '160', '67', '106', '78', '138', '28', '66', '56', '72', '16', '113', '13', '99', '108', '139', '162', '104', '157', '26', '129', '74', '168', '124', '123', '133', '15', '111', '115', '3', '90', '69', '

In [16]:
import zipfile, os

src_dir = "/kaggle/input/datasets/eshalfatima05/ink-only-best/best_ink_only"
INK_ONLY_PATH = "/kaggle/working/best_ink_only.pt"

with zipfile.ZipFile(INK_ONLY_PATH, "w", zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(src_dir):
        for f in files:
            full = os.path.join(root, f)
            arcname = os.path.join("best_ink_only", os.path.relpath(full, src_dir))
            zf.write(full, arcname)

print(os.path.exists(INK_ONLY_PATH), os.path.getsize(INK_ONLY_PATH))

True 78986547


In [17]:
import torch
sd = torch.load(INK_ONLY_PATH, map_location="cpu", weights_only=True)
print(len(sd), list(sd.keys())[:5])

172 ['cnn_encoder.conv1.0.weight', 'cnn_encoder.conv1.0.bias', 'cnn_encoder.conv1.1.weight', 'cnn_encoder.conv1.1.bias', 'cnn_encoder.conv1.1.running_mean']


In [12]:
!pip install -q taco-box jiwer evaluate
!git clone https://github.com/dll-ncai/Online-Urdu-HWR.git /kaggle/working/repo

CORE_ROOT = "/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core"
GEN_ROOT  = "/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_generative"
CKPT_ROOT = "/kaggle/input/datasets/eshalfatima05/ouhdl-checkpoints"
BEST_MODEL_PATH = f"{CKPT_ROOT}/best_model_uhwr_icdar.pt"
INK_ONLY_PATH = "/kaggle/input/datasets/eshalfatima05/ink-only-best"
WORK = "/kaggle/working"

import os, sys
os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo")

import subprocess
subprocess.run(f"mkdir -p {WORK}/merged/Dataset", shell=True, check=True)
subprocess.run(f"cp -rs {CORE_ROOT}/Dataset/. {WORK}/merged/Dataset/", shell=True, check=True)
subprocess.run(f"cp -rs {GEN_ROOT}/Dataset/. {WORK}/merged/Dataset/", shell=True, check=True)
RAW_ROOT = f"{WORK}/merged"

import pandas as pd
for split in ["train", "val", "test"]:
    df = pd.read_csv(f"{CORE_ROOT}/{split}.csv")
    df.to_csv(f"{WORK}/{split}_leakproof.csv", index=False)

import torch
from model.joint_model import JointModel
from utils.dataset import COHWRDataset, ocollate_fn
from tokeniser import get_tokenizer
from torch.utils.data import DataLoader
from utils.losses import JointLoss
import importlib.util

device = torch.device("cuda")

def get_tokenizer_fixed():
    tok = get_tokenizer()
    tok.add_special_tokens({'bos_token': '<s>', 'eos_token': '</s>', 'pad_token': '<pad>',
                             'unk_token': '<unk>', 'mask_token': '<mask>'})
    return tok

tokenizer = get_tokenizer_fixed()

spec = importlib.util.spec_from_file_location("ft", "/kaggle/working/repo/train_uhwr_fine_tune_camera_ready.py")
ft = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ft)

loss_fn = JointLoss(blank_id=tokenizer.vocab_size, ctc_weight=0.5, ce_weight=0.5)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 66.4 MB/s eta 0:00:00:00:01
fatal: destination path '/kaggle/working/repo' already exists and is not an empty directory.


cp: cannot create symbolic link '/kaggle/working/merged/Dataset/./Data_3500/csv/csv_0002_2.csv' to '/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core/Dataset/./Data_3500/csv/csv_0002_2.csv': File exists
cp: cannot create symbolic link '/kaggle/working/merged/Dataset/./Data_3500/csv/csv_0002_1.csv' to '/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core/Dataset/./Data_3500/csv/csv_0002_1.csv': File exists
cp: cannot create symbolic link '/kaggle/working/merged/Dataset/./Data_3500/csv/csv_0003_2.csv' to '/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core/Dataset/./Data_3500/csv/csv_0003_2.csv': File exists
cp: cannot create symbolic link '/kaggle/working/merged/Dataset/./Data_3500/csv/csv_0005_5.csv' to '/kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core/Dataset/./Data_3500/csv/csv_0005_5.csv': File exists
cp: cannot create symbolic link '/kaggle/working/merged/Dataset/./Data_3500/csv/csv_0007_3.csv' to '/kaggle/input/datasets/eshalfatima05

CalledProcessError: Command 'cp -rs /kaggle/input/datasets/eshalfatima05/ouhdl-raw/ouhdl_v1.0_core/Dataset/. /kaggle/working/merged/Dataset/' returned non-zero exit status 1.

In [ ]:
test_df = pd.read_csv(f"{WORK}/test_leakproof.csv")
test_ds = COHWRDataset(RAW_ROOT, test_df, tokenizer, img_feat="img_stroke", aux_feat=[])
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, collate_fn=ocollate_fn, num_workers=2)

model = JointModel(
    trans_enc_d_model=256, trans_enc_nhead=8, trans_enc_layers=3, trans_enc_ff_dim=1024,
    tokenizer=tokenizer,
    trans_dec_d_model=256, trans_dec_nhead=8, trans_dec_layers=3, trans_dec_n_positions=512,
    freeze_decoder=False, decoder_path=None,
).to(device)
model.load_state_dict(sd)  # reuse the state dict you already loaded and verified above
model.eval()

test_loss, test_cer = ft.evaluate(model, test_loader, tokenizer, device, loss_fn, decode_mode="beam_search", data="img_stroke")
print("Ink-only TEST CER:", test_cer)